# LatentMAS Instrumented Run — Kaggle

Runs `instrumented_run.py` and saves all activations needed for the 23 offline interpretability experiments.

**Output layout** (saved to `/kaggle/working/activations/`):
```
activations/
  wa_matrix.pt                    ← W_a alignment matrix (once)
  metadata.json                   ← run config
  gsm8k/
    metadata.csv                  ← per-example correctness table
    example_0000/
      latent_thoughts.pt          ← [A, m, D] pre/post W_a
      latent_per_layer.pt         ← [A, m, L+1, D] all layers
      prompt_hidden.pt            ← last 64 prompt token hidden states
      kv_latent.pt                ← KV cache at latent positions
      text_outputs.json           ← decoded text + top-5 vocab tokens
      metadata.json               ← correct / gold / pred
  arc_challenge/  ...same structure...
  mbppplus/       ...same structure...
```

**Download**: after the run finishes, go to the Kaggle notebook **Output** tab → click the folder → **Download**. For large outputs use the zip cell at the bottom.

Run is **resumable**: re-run any cell after interruption — existing `example_XXXX/` folders are skipped automatically.

## 1 — Install dependencies

In [ ]:
!pip install -q transformers accelerate datasets tqdm huggingface_hub

## 2 — Upload repo code

Two options:

**Option A** — clone from GitHub (if repo is public):
```python
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /kaggle/working/repo
%cd /kaggle/working/repo
```

**Option B** — upload as a Kaggle Dataset: zip your local repo, upload it at kaggle.com/datasets/new, then add it to this notebook via **+ Add Data**. It will appear at `/kaggle/input/YOUR_DATASET_NAME/`.

The cell below uses Option B — adjust the path to match your dataset name.

In [ ]:
import os, sys

# ── CHANGE THIS to wherever your repo code lives ──────────────────────────────
REPO_PATH = "/kaggle/input/latentmas-interp"   # Option B: Kaggle Dataset
# REPO_PATH = "/kaggle/working/repo"           # Option A: git clone
# ──────────────────────────────────────────────────────────────────────────────

assert os.path.isdir(REPO_PATH), f"Repo not found at {REPO_PATH}"
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print("Working dir:", os.getcwd())
print("Files:", os.listdir("."))

## 3 — HuggingFace login (for Qwen model)

In [ ]:
from huggingface_hub import login

# Paste your HF read token here (or use Kaggle Secrets: add HF_TOKEN as a secret)
import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")   # set via Kaggle Secrets panel
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in.")
else:
    print("No HF_TOKEN found. If Qwen3-4B is gated, the run will fail. Set it in Kaggle Secrets.")

## 4 — Config

In [ ]:
OUTPUT_DIR   = "/kaggle/working/activations"
MODEL_NAME   = "Qwen/Qwen3-4B"
TASKS        = "gsm8k arc_challenge mbppplus"
MAX_SAMPLES  = 300      # per task — keeps total output under ~15GB
LATENT_STEPS = 4
MAX_TOKENS   = 2048     # judger generation length
SEED         = 42

# Extra capture flags (set to True only if you have spare disk budget)
SAVE_ATTENTION = False  # per-layer attention maps — very large, skip unless Exp 11

print(f"Output dir : {OUTPUT_DIR}")
print(f"Model      : {MODEL_NAME}")
print(f"Tasks      : {TASKS}")
print(f"Samples    : {MAX_SAMPLES} per task = {MAX_SAMPLES * len(TASKS.split())} total")
print(f"Latent m   : {LATENT_STEPS}")

## 5 — Test run (5 examples, ~5 min)

Run this first to verify the pipeline before committing to the full run.

In [ ]:
!python {REPO_PATH}/instrumented_run.py \
    --output_dir {OUTPUT_DIR}_test \
    --model_name {MODEL_NAME} \
    --tasks gsm8k \
    --latent_steps {LATENT_STEPS} \
    --max_new_tokens 512 \
    --latent_space_realign \
    --seed {SEED} \
    --test

In [ ]:
# Quick sanity check — print what was saved for example_0000
import os
test_ex = f"{OUTPUT_DIR}_test/gsm8k/example_0000"
if os.path.isdir(test_ex):
    print("Files saved:", os.listdir(test_ex))

import torch, json
lt = torch.load(f"{test_ex}/latent_thoughts.pt", map_location="cpu", weights_only=False)
print("latent_thoughts shapes:")
print("  pre_aligned :", lt["pre_aligned"].shape,  " # [agents, m, d_h]")
print("  post_aligned:", lt["post_aligned"].shape, " # [agents, m, d_h]")
print("  agents      :", lt["agents"])

with open(f"{test_ex}/metadata.json") as f:
    print("metadata:", json.load(f))

## 6 — Full run (~8–15 hours on T4)

The run saves progress as it goes — if the session times out, re-run this cell and it will resume from where it left off.

In [ ]:
attn_flag = "--save_attention" if SAVE_ATTENTION else ""

!python {REPO_PATH}/instrumented_run.py \
    --output_dir {OUTPUT_DIR} \
    --model_name {MODEL_NAME} \
    --tasks {TASKS} \
    --max_samples {MAX_SAMPLES} \
    --latent_steps {LATENT_STEPS} \
    --max_new_tokens {MAX_TOKENS} \
    --latent_space_realign \
    --seed {SEED} \
    --storage_warn_gb 17.0 \
    --check_storage_every 25 \
    --verify_every 10 \
    {attn_flag}

## 7 — Storage report

In [ ]:
import os
from pathlib import Path

def du(path):
    total = sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file())
    return total / (1024**3)

root = Path(OUTPUT_DIR)
print(f"Total: {du(root):.2f} GB")
for task_dir in sorted(root.iterdir()):
    if task_dir.is_dir():
        n_examples = len([d for d in task_dir.iterdir() if d.is_dir()])
        print(f"  {task_dir.name}: {du(task_dir):.2f} GB  ({n_examples} examples)")

# Accuracy summary from metadata CSVs
import csv
print()
for task_dir in sorted(root.iterdir()):
    csv_path = task_dir / "metadata.csv"
    if not csv_path.exists():
        continue
    rows = list(csv.DictReader(open(csv_path)))
    n = len(rows)
    correct = sum(1 for r in rows if r.get("correct", "").lower() == "true")
    print(f"  {task_dir.name}: {correct}/{n} correct ({100*correct/n:.1f}%)")

## 8 — Zip for download

Kaggle lets you download individual files from the **Output** tab. For the full activations folder (likely 10–18 GB), zip it first — Kaggle supports downloading zip files up to 20 GB.

**After zipping**: go to your notebook page → **Output** tab → find `activations.zip` → click the download icon.

Alternatively, use the Kaggle CLI locally:
```bash
kaggle kernels output YOUR_USERNAME/YOUR_NOTEBOOK_NAME -p ./downloaded/
```

In [ ]:
# Zip the activations folder
# This may take 5–10 min depending on size
import shutil, os

zip_path = "/kaggle/working/activations"
out_zip  = "/kaggle/working/activations"   # shutil adds .zip automatically

print("Zipping... (this may take a few minutes)")
shutil.make_archive(out_zip, "zip", "/kaggle/working", "activations")
size_gb = os.path.getsize(out_zip + ".zip") / (1024**3)
print(f"Done: activations.zip  ({size_gb:.2f} GB)")
print("Download from: Notebook page → Output tab → activations.zip → download icon")

## 9 — How to load locally (quick reference)

After downloading and unzipping `activations.zip`, load any example like this:

```python
import torch, json, pandas as pd
from pathlib import Path

ROOT = Path("activations")

# Per-task accuracy table
df = pd.read_csv(ROOT / "gsm8k" / "metadata.csv")
print(df[["example_id", "correct", "prediction", "gold"]].head())

# Load one example
ex = ROOT / "gsm8k" / "example_0000"

lt  = torch.load(ex / "latent_thoughts.pt",  map_location="cpu", weights_only=False)
ll  = torch.load(ex / "latent_per_layer.pt", map_location="cpu", weights_only=False)
ph  = torch.load(ex / "prompt_hidden.pt",    map_location="cpu", weights_only=False)
kv  = torch.load(ex / "kv_latent.pt",        map_location="cpu", weights_only=False)

# lt["pre_aligned"]  : [A, m, D]      — hidden states before W_a  (Exp 1, 8, 26)
# lt["post_aligned"] : [A, m, D]      — injected latent thoughts  (Exp 2, 3, 6, 9, 10, 15–27)
# ll["hidden_per_layer"] : [A, m, L+1, D]  — all layers          (Exp 5, 7, 11, 19, 20)
# ph["planner"]      : [64, D]         — last 64 prompt tokens    (Exp 3, 9, 24)
# kv["planner"]      : list of (k,v) per layer                    (Exp 6, 11, 25)

# W_a
wa_data = torch.load(ROOT / "wa_matrix.pt", map_location="cpu", weights_only=False)
W_a = wa_data["W_a"]   # [D, D]
```